##### **Step 1**
**Data Preparation**
four types of problems (100 for each one):
1. missing value
- put NaN in a cell, e.g put NaN in the client cell. 
- using the pandas.isna() - missing client

2. format disunity
- transform abc to ABC
- standardize with str.strip()/ df["client"] = df["client"].str.strip()

3. date format disunity
- different formats: 2025-03-15/ 15/03/2025
- transform to datetime/ pd.to_datetime()

4. unmatched invoice amount
- suppose to be balance = total - discount + tax
- recalculate and mark the indifference as Amount Mismatch


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# ==========================================
# STEP 1: DATA PREPARATION
# ==========================================

# 1. document path
source_path = Path("/Users/xiao/Desktop/newest_invoices_data.csv")
output_dir = Path("/Users/xiao/Desktop/processed_invoices_data.csv")
output_dir.mkdir(parents=True, exist_ok=True)

# 2. original data
df = pd.read_csv(
    source_path,
    dtype={"id_invoice": str}
)

# 3. random 5000 records
# random_state keeps the same sample
reference = (
    df.sample(n=5000, random_state=42)
      .reset_index(drop=True)
)

# create the record_id
reference.insert(
    0,
    "record_id",
    range(1, len(reference) + 1)
)

# store the reference
reference.to_csv(
    output_dir / "invoices_reference.csv",
    index=False
)

# 4. create the copy
dirty = reference.copy()
rng = np.random.default_rng(42)

error_log = []

# 4 types problem, 100 for each
error_types = [
    "missing_client",
    "text_format",
    "date_format",
    "amount_error"
]

selected = rng.choice(
    dirty.index.to_numpy(),
    size=400,
    replace=False
)

for error_type, indices in zip(
    error_types,
    np.array_split(selected, 4)
):
    for idx in indices:
        record_id = int(dirty.at[idx, "record_id"])

        if error_type == "missing_client":
            column = "client"
            old_value = dirty.at[idx, column]
            dirty.at[idx, column] = np.nan

        elif error_type == "text_format":
            column = "client"
            old_value = dirty.at[idx, column]
            dirty.at[idx, column] = (
                "  " + str(old_value).upper() + "  "
            )

        elif error_type == "date_format":
            column = "issuedDate"
            old_value = dirty.at[idx, column]

            parsed = pd.to_datetime(
                old_value,
                errors="coerce"
            )


            if pd.isna(parsed):
                continue

            dirty.at[idx, column] = (
                parsed.strftime("%d/%m/%Y")
            )

        elif error_type == "amount_error":
            column = "balance"
            old_value = dirty.at[idx, column]

            dirty.at[idx, column] = (
                float(old_value) + 50
            )

        error_log.append({
            "record_id": record_id,
            "error_type": error_type,
            "column": column,
            "original_value": old_value,
            "modified_value": dirty.at[idx, column]
        })

# 5. save dirty data
dirty.to_csv(
    output_dir / "invoices_dirty.csv",
    index=False
)

# 6. store error terms
log_df = pd.DataFrame(error_log)

log_df.to_csv(
    output_dir / "injected_errors.csv",
    index=False
)

# 7. output
print("Original rows:", len(df))
print("Sampled rows:", len(reference))
print("Dirty rows:", len(dirty))
print("Injected changes:", len(log_df))

print("\nInjected error types:")
print(log_df["error_type"].value_counts())

print("\nFiles saved to:")
print(output_dir)


Original rows: 10000
Sampled rows: 5000
Dirty rows: 5000
Injected changes: 400

Injected error types:
error_type
missing_client    100
text_format       100
date_format       100
amount_error      100
Name: count, dtype: int64

Files saved to:
/Users/xiao/Desktop/processed_invoices_data.csv


##### **Step 2**
**Data Cleaning and Validation**
1. standardize: name/ date
2. validate: missing value/ amount calculation
3. flag: error can not automatically fixed
4. export: export the data and error report

In [5]:

from pathlib import Path
import pandas as pd
import numpy as np

# ==========================================
# STEP 2: DATA CLEANING & VALIDATION
# ==========================================

data_dir = Path("/Users/xiao/Desktop/processed_invoices_data.csv")

# 1. Load dirty data
df = pd.read_csv(
    data_dir / "invoices_dirty.csv",
    dtype={"id_invoice": str}
)

# 2. Standardize client names
# Remove extra spaces and standardize capitalization
df["client"] = (
    df["client"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 3. Standardize dates
# Support both YYYY-MM-DD and DD/MM/YYYY
def parse_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    if "/" in value:
        return pd.to_datetime(
            value, format="%d/%m/%Y", errors="coerce"
        )

    return pd.to_datetime(
        value, format="%Y-%m-%d", errors="coerce"
    )


df["issuedDate"] = df["issuedDate"].apply(parse_date)
df["dueDate"] = df["dueDate"].apply(parse_date)

# 4. Validate financial amounts
amount_columns = ["total", "discount", "tax", "balance"]

for col in amount_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

expected_balance = (
    df["total"] - df["discount"] + df["tax"]
)

# Use tolerance for floating-point comparisons
df["amount_mismatch"] = ~np.isclose(
    df["balance"],
    expected_balance,
    atol=0.01,
    rtol=0,
    equal_nan=False
)

# 5. Flag missing or invalid fields
df["missing_client"] = df["client"].isna()
df["invalid_date"] = (
    df["issuedDate"].isna() | df["dueDate"].isna()
)

df["invalid_amount"] = df[amount_columns].isna().any(axis=1)

# 6. Create overall validation status
flag_columns = [
    "missing_client",
    "invalid_date",
    "invalid_amount",
    "amount_mismatch"
]

df["validation_status"] = np.where(
    df[flag_columns].any(axis=1),
    "Review Required",
    "Valid"
)

# 7. Create exception report
exceptions = df[
    df["validation_status"] == "Review Required"
].copy()

# 8. Export results
df.to_csv(
    data_dir / "invoices_cleaned.csv",
    index=False
)

exceptions.to_csv(
    data_dir / "exception_report.csv",
    index=False
)

# 9. Display summary
print("Total records:", len(df))
print("Valid records:", (df["validation_status"] == "Valid").sum())
print("Records requiring review:", len(exceptions))

print("\nValidation issues:")
for col in flag_columns:
    print(col, ":", df[col].sum())

print("\nFiles saved to:", data_dir)


Total records: 5000
Valid records: 4800
Records requiring review: 200

Validation issues:
missing_client : 100
invalid_date : 0
invalid_amount : 0
amount_mismatch : 100

Files saved to: /Users/xiao/Desktop/processed_invoices_data.csv


In [10]:
import pandas as pd
from pathlib import Path

data_dir = Path("/Users/xiao/Desktop/processed_invoices_data.csv")

df = pd.read_csv(data_dir / "invoices_cleaned.csv")

# Generate a unique invoice number for each record
df["invoice_number"] = df["record_id"].apply(
    lambda x: f"INV2026-{x:06d}"
)

df.to_csv(data_dir / "invoices_cleaned.csv", index=False)

##### **Step 3: Invoice Lookup**
1. Search invoices using the last digits of the invoice number.
2. Display matching records, financial details, and validation status.
3. Flag invoices requiring further review.

In [13]:

import pandas as pd
from pathlib import Path

# ==========================================
# STEP 3: INVOICE LOOKUP BY TAIL NUMBER
# ==========================================

# 1. Load cleaned invoice data
data_dir = Path("/Users/xiao/Desktop/processed_invoices_data.csv")

df = pd.read_csv(
    data_dir / "invoices_cleaned.csv",
    dtype={"invoice_number": str}
)

# 2. Define invoice lookup function
def search_invoice(tail_number):
    query = str(tail_number).strip()

    results = df[
        df["invoice_number"].str.endswith(
            query, na=False
        )
    ].copy()

    return results


# 3. Select columns to display
display_columns = [
    "record_id",
    "invoice_number",
    "client",
    "issuedDate",
    "dueDate",
    "total",
    "balance",
    "invoiceStatus",
    "validation_status"
]

# 4. Repeated invoice lookup
while True:
    tail_number = input(
        "\nEnter the last 4 digits of invoice number (or 'end' to exit): "
    ).strip()

    # Exit the program
    if tail_number.lower() == "end":
        print("Invoice lookup ended.")
        break

    # Handle empty input
    if not tail_number:
        print("Please enter the last 4 digits of the invoice number.")
        continue

    # Search invoices
    results = search_invoice(tail_number)

    if results.empty:
        print("No matching invoices found.")
        continue

    # Display matching records
    print(f"\nFound {len(results)} matching records:\n")
    print(results[display_columns].to_string(index=False))

    # Count records requiring review
    review_count = (
        results["validation_status"] == "Review Required"
    ).sum()

    print(f"\nRecords requiring review: {review_count}")



Found 500 matching records:

 record_id invoice_number    client issuedDate    dueDate   total  balance invoiceStatus validation_status
         1 INV2026-000001 COMPANY E 2023-02-21 2023-03-16 6449.69  5662.86          Paid             Valid
        11 INV2026-000011 COMPANY D 2022-12-03 2022-12-20 3334.59  2982.93          Paid             Valid
        21 INV2026-000021 COMPANY B 2023-04-03 2023-04-16  957.19   867.46       Pending             Valid
        31 INV2026-000031 COMPANY C 2023-03-29 2023-04-23 2889.26  2515.78          Paid             Valid
        41 INV2026-000041 COMPANY E 2023-07-07 2023-07-17 2463.28  2225.34          Paid             Valid
        51 INV2026-000051 COMPANY B 2022-12-17 2022-12-28 2659.37  2475.22       Pending             Valid
        61 INV2026-000061 COMPANY E 2023-06-30 2023-07-12 2102.16  2099.81       Pending             Valid
        71 INV2026-000071 COMPANY A 2023-01-22 2023-01-31 4625.94  4241.44       Overdue             Valid
       

##### **Step 4: using SQL for Business Reporting**
1. how many invoices for each state (Paid, Pending, Overdue)
2. each invoice state corresponds to how many amounts
3. the customer with highest amount
4. the customer with most records to review


In [16]:

import pandas as pd
import duckdb
from pathlib import Path

# ==========================================
# STEP 4: SQL BUSINESS ANALYSIS
# ==========================================

# 1. Load cleaned invoice data
data_dir = Path("/Users/xiao/Desktop/processed_invoices_data.csv")

invoices = pd.read_csv(
    data_dir / "invoices_cleaned.csv",
    dtype={"invoice_number": str}
)

# 2. Invoice summary by status
status_summary = duckdb.sql("""
    SELECT
        invoiceStatus,
        COUNT(*) AS invoice_count,
        ROUND(SUM(balance), 2) AS total_amount
    FROM invoices
    GROUP BY invoiceStatus
    ORDER BY invoice_count DESC
""").df()

print("\n=== Invoice Summary by Status ===")
print(status_summary.to_string(index=False))


# 3. Top 10 clients by invoice amount
client_summary = duckdb.sql("""
    SELECT
        client,
        COUNT(*) AS invoice_count,
        ROUND(SUM(balance), 2) AS total_amount
    FROM invoices
    WHERE client IS NOT NULL
    GROUP BY client
    ORDER BY total_amount DESC
    LIMIT 10
""").df()

print("\n=== Top 10 Clients by Invoice Amount ===")
print(client_summary.to_string(index=False))


# 4. Invoices requiring manual review
review_summary = duckdb.sql("""
    SELECT
        client,
        COUNT(*) AS review_count
    FROM invoices
    WHERE validation_status = 'Review Required'
    GROUP BY client
    ORDER BY review_count DESC
""").df()

print("\n=== Records Requiring Review by Client ===")
print(review_summary.to_string(index=False))


# 5. Overdue Invoice Analysis
overdue_summary = duckdb.sql("""
    SELECT
        client,
        COUNT(*) AS overdue_count,
        ROUND(SUM(balance), 2) AS overdue_amount
    FROM invoices
    WHERE invoiceStatus = 'Overdue'
    GROUP BY client
    ORDER BY overdue_amount DESC
""").df()

print("\n=== Overdue Invoice Analysis ===")
print(overdue_summary.to_string(index=False))


# 6. Monthly Invoice Analysis
monthly_summary = duckdb.sql("""
    SELECT
        STRFTIME(
            TRY_CAST(issuedDate AS DATE),
            '%Y-%m'
        ) AS invoice_month,
        COUNT(*) AS invoice_count,
        ROUND(SUM(balance), 2) AS total_amount
    FROM invoices
    WHERE TRY_CAST(issuedDate AS DATE) IS NOT NULL
    GROUP BY invoice_month
    ORDER BY invoice_month
""").df()

print("\n=== Monthly Invoice Analysis ===")
print(monthly_summary.to_string(index=False))


# 7. Data Quality Summary
quality_summary = duckdb.sql("""
    SELECT
        COUNT(*) AS total_records,
        COUNT(*) FILTER (
            WHERE validation_status = 'Review Required'
        ) AS review_required,
        COUNT(*) FILTER (
            WHERE missing_client = TRUE
        ) AS missing_client_count,
        COUNT(*) FILTER (
            WHERE amount_mismatch = TRUE
        ) AS amount_mismatch_count
    FROM invoices
""").df()

print("\n=== Data Quality Summary ===")
print(quality_summary.to_string(index=False))


# 8. Export SQL reports
status_summary.to_csv(
    data_dir / "status_summary.csv",
    index=False
)

client_summary.to_csv(
    data_dir / "client_summary.csv",
    index=False
)

review_summary.to_csv(
    data_dir / "review_summary.csv",
    index=False
)

overdue_summary.to_csv(
    data_dir / "overdue_summary.csv",
    index=False
)

monthly_summary.to_csv(
    data_dir / "monthly_summary.csv",
    index=False
)

quality_summary.to_csv(
    data_dir / "quality_summary.csv",
    index=False
)

print("\nAll 6 SQL reports exported successfully.")



=== Invoice Summary by Status ===
invoiceStatus  invoice_count  total_amount
         Paid           1694    5334733.96
      Overdue           1663    5195587.06
      Pending           1643    5341947.78

=== Top 10 Clients by Invoice Amount ===
   client  invoice_count  total_amount
COMPANY B           1011    3255524.18
COMPANY C           1022    3217294.20
COMPANY E            975    3063239.79
COMPANY D            960    3053421.11
COMPANY A            932    2951590.66

=== Records Requiring Review by Client ===
   client  review_count
     None           100
COMPANY E            22
COMPANY B            22
COMPANY C            21
COMPANY D            19
COMPANY A            16

=== Overdue Invoice Analysis ===
   client  overdue_count  overdue_amount
COMPANY C            360      1174599.89
COMPANY B            337      1095064.53
COMPANY E            345      1060977.48
COMPANY D            299       921902.67
COMPANY A            281       821995.80
     None             41 

##### **Step 5: Website**

In [18]:

from pathlib import Path
from datetime import datetime

import pandas as pd
import streamlit as st

# ==========================================
# STEP 5: STREAMLIT WEB APPLICATION
# ==========================================

data_dir = Path("/Users/xiao/Desktop/processed_invoices_data.csv")
log_path = data_dir / "verification_log.csv"

st.set_page_config(
    page_title="InvoiceFlow",
    page_icon="📄",
    layout="wide"
)

# 1. Load invoice data
@st.cache_data
def load_invoices():
    return pd.read_csv(
        data_dir / "invoices_cleaned.csv",
        dtype={"invoice_number": str}
    )


def load_verification_log():
    if log_path.exists():
        return pd.read_csv(log_path)

    return pd.DataFrame(
        columns=["record_id", "invoice_number", "verified_at"]
    )


def mark_as_verified(record):
    log = load_verification_log()
    record_id = int(record["record_id"])

    # Prevent duplicate verification
    verified_ids = set(
        pd.to_numeric(log["record_id"], errors="coerce")
        .dropna()
        .astype(int)
    )

    if record_id in verified_ids:
        return False

    new_entry = pd.DataFrame([{
        "record_id": record_id,
        "invoice_number": record["invoice_number"],
        "verified_at": datetime.now().isoformat(
            timespec="seconds"
        )
    }])

    log = pd.concat([log, new_entry], ignore_index=True)
    log.to_csv(log_path, index=False)
    return True


invoices = load_invoices()
verification_log = load_verification_log()

verified_ids = set(
    pd.to_numeric(
        verification_log["record_id"],
        errors="coerce"
    ).dropna().astype(int)
)

# 2. Page header
st.title("InvoiceFlow")
st.caption("Invoice Lookup, Validation & Business Reporting")

page = st.sidebar.radio(
    "Navigation",
    ["Invoice Lookup", "Data Validation", "Business Reports"]
)

# ==========================================
# PAGE 1: INVOICE LOOKUP
# ==========================================

if page == "Invoice Lookup":

    st.header("Invoice Lookup by Tail Number")

    tail_number = st.text_input(
        "Enter the last 4 digits of the invoice number",
        placeholder="e.g. 0125 (1–4 digits accepted)"
    ).strip()

    if tail_number:

        if not tail_number.isdigit() or len(tail_number) > 4:
            st.warning("Please enter 1 to 4 digits.")

        else:
            results = invoices[
                invoices["invoice_number"].str.endswith(
                    tail_number, na=False
                )
            ]

            st.write(f"Found {len(results)} matching records.")

            if results.empty:
                st.info("No matching invoices found.")

            else:
                for _, record in results.iterrows():

                    record_id = int(record["record_id"])
                    already_verified = record_id in verified_ids

                    with st.container(border=True):
                        st.subheader(record["invoice_number"])

                        col1, col2, col3 = st.columns(3)

                        col1.write(f"Client: {record['client']}")
                        col2.write(f"Amount: {record['balance']}")
                        col3.write(
                            f"Invoice Status: {record['invoiceStatus']}"
                        )

                        st.write(
                            "Data Validation:",
                            record["validation_status"]
                        )

                        if already_verified:
                            st.success("Already Verified")

                        else:
                            st.info("Not Yet Verified")

                            if st.button(
                                "Mark as Verified",
                                key=f"verify_{record_id}"
                            ):
                                if mark_as_verified(record):
                                    st.rerun()
                                else:
                                    st.warning(
                                        "This invoice is already verified."
                                    )

    st.divider()
    st.metric("Verified Invoices", len(verified_ids))


# ==========================================
# PAGE 2: DATA VALIDATION
# ==========================================

elif page == "Data Validation":

    st.header("Data Validation Summary")

    total = len(invoices)
    review_count = (
        invoices["validation_status"] == "Review Required"
    ).sum()

    valid_count = (
        invoices["validation_status"] == "Valid"
    ).sum()

    col1, col2, col3 = st.columns(3)

    col1.metric("Total Records", total)
    col2.metric("Valid Records", valid_count)
    col3.metric("Review Required", review_count)

    st.subheader("Records Requiring Review")

    exceptions = invoices[
        invoices["validation_status"] == "Review Required"
    ]

    st.dataframe(exceptions, use_container_width=True)

    st.download_button(
        "Download Exception Report",
        data=exceptions.to_csv(index=False),
        file_name="exception_report.csv",
        mime="text/csv"
    )


# ==========================================
# PAGE 3: BUSINESS REPORTS
# ==========================================

elif page == "Business Reports":

    st.header("SQL Business Reports")

    reports = {
        "Invoice Status Summary": "status_summary.csv",
        "Top Clients": "client_summary.csv",
        "Review Summary": "review_summary.csv",
        "Overdue Invoice Analysis": "overdue_summary.csv",
        "Monthly Invoice Analysis": "monthly_summary.csv",
        "Data Quality Summary": "quality_summary.csv"
    }

    selected_report = st.selectbox(
        "Select a report",
        list(reports.keys())
    )

    report_path = data_dir / reports[selected_report]

    if report_path.exists():
        report = pd.read_csv(report_path)

        st.dataframe(report, use_container_width=True)

        st.download_button(
            "Download Report",
            data=report.to_csv(index=False),
            file_name=reports[selected_report],
            mime="text/csv"
        )

    else:
        st.warning("Report file not found. Run Step 4 first.")


2026-11-23 23:42:37.405 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 23:42:37.406 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-23 23:42:37.406 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-11-23 23:42:37.406 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 23:42:37.416 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 23:42:37.442 
  command:

    streamlit run /Users/xiao/python-projects/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-11-23 23:42:37.442 Thr